In [5]:
import pandas as pd
import numpy as np
import time
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [6]:
## CONFIG
USE_PCA = True
PCA_VARIANCE = 0.5

In [7]:
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

X_train = train_df.drop('label', axis=1).values
y_train = train_df['label'].values
X_test = test_df.drop('label', axis=1).values
y_test = test_df['label'].values

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {np.unique(y_train)}")

Train: (38, 7129), Test: (34, 7129)
Classes: [0 1]


In [8]:
from sklearn.decomposition import PCA

if USE_PCA:
    pca = PCA(n_components=PCA_VARIANCE)
    X_train_final = pca.fit_transform(X_train)
    X_test_final = pca.transform(X_test)
    print(f"PCA: {X_train.shape[1]} -> {pca.n_components_} features")
    print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")
else:
    X_train_final = X_train
    X_test_final = X_test
    print(f"No PCA — using all {X_train.shape[1]} features")

PCA: 7129 -> 7 features
Total variance explained: 0.5030


In [9]:
def count_params(model):
    name = type(model).__name__
    if name == 'LogisticRegression':
        return model.coef_.size + model.intercept_.size
    elif name == 'SVC':
        return model.support_vectors_.size + model.intercept_.size + model.dual_coef_.size
    elif name == 'GaussianNB':
        return model.theta_.size + model.var_.size + model.class_prior_.size
    elif name in ('RandomForestClassifier', 'GradientBoostingClassifier'):
        return sum(t.tree_.node_count for e in model.estimators_ for t in (e if hasattr(e, '__iter__') else [e]))
    elif name == 'KNeighborsClassifier':
        return model._fit_X.size
    elif name == 'DecisionTreeClassifier':
        return model.tree_.node_count
    elif name == 'MLPClassifier':
        return sum(w.size for w in model.coefs_) + sum(b.size for b in model.intercepts_)
    elif name == 'XGBClassifier':
        booster = model.get_booster()
        trees = booster.trees_to_dataframe()
        return len(trees)
    elif name == 'LGBMClassifier':
        return model.booster_.num_trees()
    return 'N/A'

In [10]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'SVM (Linear)': SVC(kernel='linear', random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'MLP': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000, random_state=42),
}

In [11]:
results = []

num_features = X_train_final.shape[1]
pca_variance = round(pca.explained_variance_ratio_.sum(), 4) if USE_PCA else 'N/A'

for name, model in models.items():
    start = time.time()
    model.fit(X_train_final, y_train)
    train_time = time.time() - start

    train_acc = accuracy_score(y_train, model.predict(X_train_final))
    test_acc = accuracy_score(y_test, model.predict(X_test_final))
    params = count_params(model)

    results.append({
        'Model': name,
        'Num Features': num_features,
        'PCA Variance': pca_variance,
        'Parameters': params,
        'Train Time (s)': round(train_time, 4),
        'Train Accuracy': round(train_acc, 4),
        'Test Accuracy': round(test_acc, 4),
    })
    print(f"{name:25s} | Features: {num_features:>5} | PCA Var: {str(pca_variance):>6s} | Params: {str(params):>10s} | Time: {train_time:.4f}s | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

results_df = pd.DataFrame(results)
results_df

Logistic Regression       | Features:     7 | PCA Var:  0.503 | Params:          8 | Time: 0.0198s | Train Acc: 1.0000 | Test Acc: 0.6471
SVM (RBF)                 | Features:     7 | PCA Var:  0.503 | Params:        209 | Time: 0.0004s | Train Acc: 0.9474 | Test Acc: 0.6176
SVM (Linear)              | Features:     7 | PCA Var:  0.503 | Params:         57 | Time: 0.0020s | Train Acc: 1.0000 | Test Acc: 0.6471
Gaussian Naive Bayes      | Features:     7 | PCA Var:  0.503 | Params:         30 | Time: 0.0000s | Train Acc: 0.9737 | Test Acc: 0.6176
Random Forest             | Features:     7 | PCA Var:  0.503 | Params:       1122 | Time: 0.0703s | Train Acc: 1.0000 | Test Acc: 0.5882
Decision Tree             | Features:     7 | PCA Var:  0.503 | Params:         11 | Time: 0.0000s | Train Acc: 1.0000 | Test Acc: 0.6471
KNN (k=5)                 | Features:     7 | PCA Var:  0.503 | Params:        266 | Time: 0.0010s | Train Acc: 0.9211 | Test Acc: 0.6176
Gradient Boosting         | Featur

d:\__Projects\benchmark-qml-cml\.venv312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\__Projects\benchmark-qml-cml\.venv312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,Num Features,PCA Variance,Parameters,Train Time (s),Train Accuracy,Test Accuracy
0,Logistic Regression,7,0.503,8,0.0198,1.0000,0.6471
1,SVM (RBF),7,0.503,209,0.0004,0.9474,0.6176
2,SVM (Linear),7,0.503,57,0.0020,1.0000,0.6471
3,Gaussian Naive Bayes,7,0.503,30,0.0000,0.9737,0.6176
4,Random Forest,7,0.503,1122,0.0703,1.0000,0.5882
5,Decision Tree,7,0.503,11,0.0000,1.0000,0.6471
6,KNN (k=5),7,0.503,266,0.0010,0.9211,0.6176
7,Gradient Boosting,7,0.503,1264,0.0433,1.0000,0.5294
8,XGBoost,7,0.503,196,0.1599,0.9737,0.6765
9,LightGBM,7,0.503,1,2.2091,0.7105,0.5882


In [12]:
results_df.to_csv(f'Model_Comparison_PCA={PCA_VARIANCE}.csv', index=False)
print(f'Results saved to Model_Comparison_PCA={PCA_VARIANCE}.csv')




Results saved to Model_Comparison_PCA=0.5.csv
